# Incoming-lot risk scoring (PROTOTYPE)

**Status: proposed / prototype. Not a finished or validated system.**

This notebook uses a **synthetic** dataset (`data/mock_lots.csv`) to show the shape of one idea:
score incoming raw-material lots *before* the rapid aflatoxin test result is known, so QC can prioritise probing and sampling.
Because the data is simulated from assumptions I wrote, the metrics below show that the pipeline works. They are **not** evidence about real feed mills.

In [1]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/mock_lots.csv", parse_dates=["arrival_time"]).sort_values("arrival_time").reset_index(drop=True)
print(df.shape, "| positive rate:", round(df.aflatoxin_positive.mean(), 3))
df.groupby("material").aflatoxin_positive.agg(["count", "mean"]).round(3)

(1200, 11) | positive rate: 0.233


,count,mean
material,,
canola_meal,178,0.146
cottonseed_meal,182,0.258
maize,545,0.266
soybean_meal,295,0.210


## Features
The rapid-test reading is **excluded** from the features. It is the label's source, and including it would leak the answer.
Supplier history uses only *earlier* lots (expanding mean, shifted by one, smoothed toward the overall rate).

In [2]:
prior = df.aflatoxin_positive.mean()
k = 5  # smoothing strength
g = df.groupby("supplier_id").aflatoxin_positive
cum_pos = g.cumsum() - df.aflatoxin_positive
cum_n = g.cumcount()
df["supplier_prior_rate"] = (cum_pos + k * prior) / (cum_n + k)

num = ["moisture_pct", "ambient_humidity_pct", "storage_days_before_arrival",
       "transit_hours", "visual_defect_pct", "supplier_prior_rate"]
cat = ["material"]
X, y = df[num + cat], df.aflatoxin_positive

## Time-based split
Train on the earlier 80% of lots, test on the latest 20%. A random split would let the model see the future.

In [3]:
cut = int(len(df) * 0.8)
Xtr, Xte, ytr, yte = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]

pre = ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
model = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]

base = Xte["moisture_pct"]  # simple rule-of-thumb baseline: wetter lot = riskier
print("Model    ROC AUC:", round(roc_auc_score(yte, p), 3), "| avg precision:", round(average_precision_score(yte, p), 3))
print("Baseline ROC AUC:", round(roc_auc_score(yte, base), 3), "| avg precision:", round(average_precision_score(yte, base), 3))
print("Test positive rate:", round(yte.mean(), 3))

Model    ROC AUC: 0.803 | avg precision: 0.671
Baseline ROC AUC: 0.743 | avg precision: 0.583
Test positive rate: 0.279


## Risk bands
Bands are for **sampling priority**, not for approving or rejecting lots.

In [4]:
te = Xte.copy()
te["risk"] = p
te["actual"] = yte.values
te["band"] = pd.cut(te.risk, [0, 0.15, 0.35, 1], labels=["low", "medium", "high"])
te.groupby("band", observed=True).agg(lots=("actual", "size"), positive_rate=("actual", "mean")).round(3)

,lots,positive_rate
band,,
low,101,0.089
medium,73,0.247
high,66,0.606


In [5]:
names = model.named_steps["pre"].get_feature_names_out()
coef = pd.Series(model.named_steps["lr"].coef_[0], index=names).sort_values(key=abs, ascending=False)
coef.head(8).round(2)

n__moisture_pct                   1.04
c__material_canola_meal          -0.63
n__ambient_humidity_pct           0.44
c__material_cottonseed_meal       0.34
n__visual_defect_pct              0.32
n__supplier_prior_rate            0.30
n__storage_days_before_arrival    0.26
n__transit_hours                  0.24
dtype: float64

## Hard-stop overlay
Safety stops are fixed rules, not model output. A positive rapid test always means HOLD. The score only decides how closely to look at everything else.

In [6]:
def lot_decision(rapid_ppb, risk, limit_ppb=20):
    if rapid_ppb >= limit_ppb:          # illustrative threshold, not a regulatory limit
        return "HOLD (rule: positive rapid test)"
    band = "high" if risk > 0.35 else "medium" if risk > 0.15 else "low"
    return f"Proceed with standard checks (risk band: {band})"

demo = df.iloc[cut:].head(5)[["lot_id", "rapid_aflatoxin_ppb"]].assign(risk=p[:5].round(2))
demo["decision"] = [lot_decision(a, b) for a, b in zip(demo.rapid_aflatoxin_ppb, demo.risk)]
demo

,lot_id,rapid_aflatoxin_ppb,risk,decision
960,LOT-0961,2.1,0.56,Proceed with standard checks (risk band: high)
961,LOT-0962,2.0,0.25,Proceed with standard checks (risk band: medium)
962,LOT-0963,4.7,0.45,Proceed with standard checks (risk band: high)
963,LOT-0964,2.7,0.18,Proceed with standard checks (risk band: medium)
964,LOT-0965,2.3,0.20,Proceed with standard checks (risk band: medium)


## Limitations
- The data is synthetic; the model recovers relationships I built in, so these metrics say nothing about real mills.
- The 20 ppb threshold and the risk-band cut-offs are placeholders.
- A real version would need lot-level records from actual mills, permission to use them, and validation against QC outcomes.
- The score supports sampling decisions only. It never overrides a test result or a QC hold.